# Kaggle Master Plan: SHL Grammar Scoring

Ultimate Hybrid Pipeline with Parselmouth Voice Quality, Syntactic Dependency Trees, WavLM-Large, BGE-Large, Expanded Stacking, and Error Analysis.

In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install -q pandas numpy scikit-learn librosa soundfile openai-whisper spacy sentence-transformers catboost xgboost lightgbm transformers praat-parselmouth textstat
!python -m spacy download en_core_web_sm
!apt-get update && apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 59.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' o

In [ ]:
# 2. CONFIGURATION & MOUNT
import os
import warnings
import pandas as pd
import numpy as np
import torch
import librosa
import parselmouth
from parselmouth.praat import call
import textstat
import whisper
import spacy
from transformers import Wav2Vec2FeatureExtractor, WavLMModel
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = "/content/drive/MyDrive/Dataset_Final"
os.makedirs("cache", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

df_train = pd.read_csv(f"{DATASET_DIR}/train.csv")
df_test = pd.read_csv(f"{DATASET_DIR}/test.csv")
df_train['audio_path'] = df_train['filename'].apply(lambda x: f"{DATASET_DIR}/train/{x}")
df_test['audio_path'] = df_test['filename'].apply(lambda x: f"{DATASET_DIR}/test/{x}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dataset Loaded. Using device: {device}")

ValueError: mount failed

## Phase 1A: Advanced Voice Quality (Parselmouth & Librosa)

In [ ]:
def extract_advanced_acoustics(df, cache_file):
    if os.path.exists(cache_file): return np.load(cache_file)

    features = []
    for path in tqdm(df['audio_path'], desc="Voice Quality"):
        try:
            # Librosa basics
            y, sr = librosa.load(path, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)
            rms = librosa.feature.rms(y=y)[0]
            silence_ratio = np.mean(rms < 0.01)
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            tempo_val = tempo[0] if isinstance(tempo, np.ndarray) else tempo

            # Praat / Parselmouth (Jitter, Shimmer, HNR, Pitch)
            snd = parselmouth.Sound(path)
            pitch = snd.to_pitch()
            pitch_vals = pitch.selected_array['frequency']
            pitch_vals = pitch_vals[pitch_vals > 0]
            p_mean = np.mean(pitch_vals) if len(pitch_vals) > 0 else 0
            p_std = np.std(pitch_vals) if len(pitch_vals) > 0 else 0

            harmonicity = snd.to_harmonicity()
            hnr_vals = harmonicity.values[harmonicity.values > 0]
            hnr = np.mean(hnr_vals) if len(hnr_vals) > 0 else 0

            pointProcess = call(snd, "To PointProcess (periodic, cc)", 75, 500)
            jitter = call(pointProcess, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
            shimmer = call([snd, pointProcess], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

            # Handle Praat returning undefined/nan
            jitter = 0 if np.isnan(jitter) else jitter
            shimmer = 0 if np.isnan(shimmer) else shimmer

            feat = np.concatenate([mfcc, [silence_ratio, tempo_val, p_mean, p_std, hnr, jitter, shimmer]])
            features.append(feat)
        except Exception as e:
            features.append(np.zeros(13 + 7))

    feats = np.array(features)
    np.save(cache_file, feats)
    return feats

print("Extracting Voice Quality Features...")
X_train_ac = extract_advanced_acoustics(df_train, "cache/train_acoustics_master.npy")
X_test_ac = extract_advanced_acoustics(df_test, "cache/test_acoustics_master.npy")

## Phase 1B: WavLM-Large Embeddings (Mean + Std + Max Pooling)

In [ ]:
def extract_wavlm_large(df, cache_file):
    if os.path.exists(cache_file): return np.load(cache_file)

    processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-large")
    model = WavLMModel.from_pretrained("microsoft/wavlm-large").to(device)

    features = []
    for path in tqdm(df['audio_path'], desc="WavLM Large"):
        try:
            y, sr = librosa.load(path, sr=16000)
            inputs = processor(y, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                hidden_states = model(**inputs).last_hidden_state.squeeze()

            if len(hidden_states.shape) == 1:
                hidden_states = hidden_states.unsqueeze(0)

            mean_pool = hidden_states.mean(dim=0).cpu().numpy()
            std_pool = hidden_states.std(dim=0).cpu().numpy()
            max_pool = hidden_states.max(dim=0)[0].cpu().numpy()

            features.append(np.concatenate([mean_pool, std_pool, max_pool]))
        except Exception as e:
            features.append(np.zeros(1024 * 3))

    feats = np.array(features)
    np.save(cache_file, feats)
    return feats

print("Extracting WavLM Large Features...")
X_train_sp = extract_wavlm_large(df_train, "cache/train_wavlm_large.npy")
X_test_sp = extract_wavlm_large(df_test, "cache/test_wavlm_large.npy")

## Phase 1C: BGE-Large Text Embeddings & Syntactic Tree Depth

In [ ]:
def extract_text_master(df, cache_prefix):
    txt_cache = f"cache/{cache_prefix}_transcripts_master.csv"
    emb_cache = f"cache/{cache_prefix}_text_emb_master.npy"
    gram_cache = f"cache/{cache_prefix}_grammar_master.npy"

    if os.path.exists(emb_cache) and os.path.exists(gram_cache):
        return np.load(gram_cache), np.load(emb_cache)

    asr_model = whisper.load_model("base").to(device)
    transcripts = []
    for path in tqdm(df['audio_path'], desc="Whisper"):
        try: transcripts.append(asr_model.transcribe(path, fp16=(device=="cuda"))["text"].strip())
        except: transcripts.append("")

    df['transcript'] = transcripts
    df[['filename', 'transcript']].to_csv(txt_cache, index=False)

    # BGE-Large Embeddings
    emb_model = SentenceTransformer("BAAI/bge-large-en-v1.5")
    emb = emb_model.encode(transcripts, show_progress_bar=True)
    np.save(emb_cache, emb)

    nlp = spacy.load("en_core_web_sm")
    grammar = []
    filler_words = ['uh', 'um', 'like', 'you know', 'so']

    for t in tqdm(transcripts, desc="Syntax & Lexical Diversity"):
        doc = nlp(t)
        words = [tok for tok in doc if not tok.is_punct]

        # Dependency Tree Depth
        dep_depths = [len(list(tok.ancestors)) for tok in doc]
        avg_dep_depth = np.mean(dep_depths) if dep_depths else 0
        max_dep_depth = np.max(dep_depths) if dep_depths else 0

        # Repetitions & Fillers
        word_texts = [w.text.lower() for w in words]
        repetitions = sum(1 for i in range(1, len(word_texts)) if word_texts[i] == word_texts[i-1])
        fillers = sum(1 for w in word_texts if w in filler_words)

        # Lexical Diversity / Readability via textstat
        flesch = textstat.flesch_reading_ease(t) if t else 0
        lexicon_ct = textstat.lexicon_count(t) if t else 0

        wc = len(words)
        grammar.append([
            wc, len(list(doc.sents)), avg_dep_depth, max_dep_depth,
            repetitions, fillers, flesch, lexicon_ct
        ])

    gram = np.array(grammar)
    np.save(gram_cache, gram)
    return gram, emb

print("Extracting Master Text Features...")
X_train_gr, X_train_te = extract_text_master(df_train, "train")
X_test_gr, X_test_te = extract_text_master(df_test, "test")

## Phase 2: PCA Component Search (Ablation Avoidance)

In [ ]:
# To avoid a slow nested CV grid search in this notebook, we perform a rapid validation split
# to find the best PCA dims for WavLM and Text Embeddings.

from sklearn.model_selection import train_test_split
X_tr_idx, X_va_idx = train_test_split(range(len(y_train)), test_size=0.2, random_state=42)

best_sp_dim, best_te_dim = 60, 40 # Fallback defaults
best_rmse = 999

print("Running rapid PCA grid search on SVR...")
for sp_dim in [40, 60, 80]:
    for te_dim in [20, 40, 60]:
        pca_sp = PCA(n_components=sp_dim, random_state=42).fit(X_train_sp[X_tr_idx])
        pca_te = PCA(n_components=te_dim, random_state=42).fit(X_train_te[X_tr_idx])

        X_tmp_tr = np.hstack([X_train_ac[X_tr_idx], X_train_gr[X_tr_idx], pca_sp.transform(X_train_sp[X_tr_idx]), pca_te.transform(X_train_te[X_tr_idx])])
        X_tmp_va = np.hstack([X_train_ac[X_va_idx], X_train_gr[X_va_idx], pca_sp.transform(X_train_sp[X_va_idx]), pca_te.transform(X_train_te[X_va_idx])])

        model = Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
        model.fit(X_tmp_tr, y_train[X_tr_idx])
        preds = np.clip(model.predict(X_tmp_va), 0, 5)
        rmse = root_mean_squared_error(y_train[X_va_idx], preds)

        if rmse < best_rmse:
            best_rmse = rmse
            best_sp_dim, best_te_dim = sp_dim, te_dim

print(f"Optimal PCA Dims Found -> WavLM: {best_sp_dim}, Text: {best_te_dim} (Validation RMSE: {best_rmse:.4f})")

pca_sp_final = PCA(n_components=best_sp_dim, random_state=42).fit(X_train_sp)
pca_te_final = PCA(n_components=best_te_dim, random_state=42).fit(X_train_te)

X_train_all = np.hstack([X_train_ac, X_train_gr, pca_sp_final.transform(X_train_sp), pca_te_final.transform(X_train_te)])
X_test_all = np.hstack([X_test_ac, X_test_gr, pca_sp_final.transform(X_test_sp), pca_te_final.transform(X_test_te)])
print("Final Combined Feature Shape:", X_train_all.shape)

## Phase 3: The Expanded OOF Stacking Ensemble

In [ ]:
def get_oof_predictions(X, y, X_test, model):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))

    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_va = scaler.transform(X_va)
        X_te = scaler.transform(X_test)

        model.fit(X_tr, y_tr)
        oof[va_idx] = model.predict(X_va)
        test_preds += model.predict(X_te) / 5.0

    return oof, test_preds

models = {
    "SVR": SVR(C=2.0, kernel='rbf', gamma='scale'),
    "CatBoost": CatBoostRegressor(iterations=500, depth=4, l2_leaf_reg=5, learning_rate=0.03, verbose=False, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=300, max_depth=4, reg_lambda=2.0, learning_rate=0.03, verbosity=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, max_depth=4, reg_lambda=2.0, learning_rate=0.03, random_state=42),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=3, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.05, l1_ratio=0.5, random_state=42)
}

oof_matrix = np.zeros((len(y_train), len(models)))
test_matrix = np.zeros((len(X_test_all), len(models)))

print("Generating Out-Of-Fold predictions for 6 Base Models...")
for i, (name, model) in enumerate(models.items()):
    oof, test_p = get_oof_predictions(X_train_all, y_train, X_test_all, model)
    oof_matrix[:, i] = oof
    test_matrix[:, i] = test_p
    score = root_mean_squared_error(y_train, oof)
    print(f"{name} OOF RMSE: {score:.4f}")

from sklearn.linear_model import BayesianRidge
print("\nTraining Meta-Model (Bayesian Ridge) on OOF predictions...")
meta_model = BayesianRidge()
meta_model.fit(oof_matrix, y_train)

stack_oof_preds = meta_model.predict(oof_matrix)
stack_rmse = root_mean_squared_error(y_train, stack_oof_preds)
print(f"** Final Master Stacking OOF RMSE: {stack_rmse:.4f} **")

final_preds = np.clip(meta_model.predict(test_matrix), 0, 5)
sub = df_test[['filename']].copy()
sub['label'] = final_preds
sub.to_csv("outputs/MASTER_SUBMISSION.csv", index=False)
print("Saved outputs/MASTER_SUBMISSION.csv")

## Phase 4: Error Analysis (The Residual Hunt)
Examine the top 30 worst predictions to figure out what feature family to build next!

In [ ]:
# Calculate absolute errors of our final OOF predictions
errors = np.abs(y_train - stack_oof_preds)

# Create an analysis dataframe
analysis_df = pd.DataFrame({
    'Filename': df_train['filename'],
    'True_Score': y_train,
    'Predicted_Score': stack_oof_preds,
    'Absolute_Error': errors
})

# Get Top 30 Worst Predictions
worst_30 = analysis_df.sort_values('Absolute_Error', ascending=False).head(30)

print("Top 30 Largest Residuals (Worst Predictions):")
display(worst_30)

print("\n-> Listen to these 30 audio files in Dataset_Final/train/.")
print("-> Ask yourself: Why did the model fail? (Heavy accent? Background noise? Very short audio?)")
print("-> Whatever the pattern is, engineer a feature for it in Phase 1!")